# JLS analyse heuristique
Analyse rapide des CSV heuristiques et export d'un README.

In [46]:
from pathlib import Path
from datetime import datetime
import pandas as pd

base_dir = Path.cwd()
out_dir = base_dir / 'analyse' / 'csv_heuristique_study'
out_dir.mkdir(parents=True, exist_ok=True)

path_lab = base_dir / 'clauses_labels_train.csv'
path_full = base_dir / 'parsed_clauses_full.csv'

df_lab = None
df_full = None

if path_lab.exists():
    df_lab = pd.read_csv(path_lab)
else:
    print(f'Missing file: {path_lab}')

if path_full.exists():
    df_full = pd.read_csv(path_full)
else:
    print(f'Missing file: {path_full}')

# Normalize R23 -> C2 in df_lab and df_full (rule codes)
for _df in [df_lab, df_full]:
    if _df is None:
        continue
    for col in _df.columns:
        if _df[col].dtype == object:
            _df[col] = _df[col].where(
                _df[col].isna(),
                _df[col].astype(str).str.replace('R23', 'C2')
            )

# Sanity checks
if df_lab is not None:
    print('df_lab shape:', df_lab.shape)
    print('df_lab columns:', list(df_lab.columns))
    if 'agent_rule' in df_lab.columns:
        print('agent_rule value_counts:')
        print(df_lab['agent_rule'].value_counts())
    if 'patient_rule' in df_lab.columns:
        print('patient_rule value_counts:')
        print(df_lab['patient_rule'].value_counts())
    if 'needs_ml_agent' in df_lab.columns:
        print('needs_ml_agent value_counts:')
        print(df_lab['needs_ml_agent'].value_counts())
    if 'needs_ml_patient' in df_lab.columns:
        print('needs_ml_patient value_counts:')
        print(df_lab['needs_ml_patient'].value_counts())

if df_full is not None:
    print('df_full shape:', df_full.shape)
    print('df_full columns:', list(df_full.columns))

# README
readme_lines = []
readme_lines.append('JLS analyse heuristique - README')
readme_lines.append('')
readme_lines.append(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
readme_lines.append(f'Path df_lab: {path_lab}')
readme_lines.append(f'Path df_full: {path_full}')

if df_lab is not None:
    readme_lines.append(f'df_lab shape: {df_lab.shape}')
else:
    readme_lines.append('df_lab shape: MISSING')

if df_full is not None:
    readme_lines.append(f'df_full shape: {df_full.shape}')
else:
    readme_lines.append('df_full shape: MISSING')

(out_dir / 'README.txt').write_text(''.join(readme_lines), encoding='utf-8')
print('Saved:', out_dir / 'README.txt')


df_lab shape: (445, 39)
df_lab columns: ['clause_id', 'action_value', 'agent label', 'patient labels', 'action_conf', 'action_rule', 'agent_value', 'agent_conf', 'agent_rule', 'patient_value', 'patient_conf', 'patient_rule', 'needs_ml_agent', 'needs_ml_patient', 'agent_candidates', 'patient_candidates', 'topic_conf', 'last_agent_conf', 'last_patient_conf', 'unknown_conf', 'has_verb', 'verb_index', 'noun_count', 'postverb_is_pt', 'postverb_pt_resolved', 'pt_is_bare', 'boundary_strength', 'contexte', 'agent_top1', 'agent_top2', 'agent_margin', 'agent_max_non_unknown', 'patient_top1', 'patient_top2', 'patient_margin', 'patient_max_non_unknown', 'needs_ml_agent_margin', 'needs_ml_patient_margin', 'reason_expansion']
agent_rule value_counts:
agent_rule
R8                  362
C2_DROP_AGENT_R8     83
Name: count, dtype: int64
patient_rule value_counts:
patient_rule
C2_DROP_WEAKER    195
R11               127
R12                66
R10                50
R13                 6
R8                

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)

def _to_bool_series(s):
    return s.astype(str).str.strip().str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)

def coverage_table(df, rule_col, needs_col, out_csv, out_png):
    df = df.copy()
    df[needs_col] = _to_bool_series(df[needs_col])

    g = df.groupby(rule_col, dropna=False)[needs_col].value_counts(dropna=False).unstack(fill_value=0)

    # use reindex to avoid boolean mask issues
    g = g.reindex(columns=[False, True], fill_value=0)
    g.columns = ["needs_ml_false", "needs_ml_true"]

    g["N"] = g["needs_ml_false"] + g["needs_ml_true"]
    g = g.reset_index().rename(columns={rule_col: "rule"})
    g = g.sort_values("N", ascending=False)

    # group rare rules
    rare = g["N"] < 10
    if rare.any():
        other = pd.DataFrame({
            "rule": ["OTHER(<10)"],
            "needs_ml_false": [g.loc[rare, "needs_ml_false"].sum()],
            "needs_ml_true": [g.loc[rare, "needs_ml_true"].sum()],
            "N": [g.loc[rare, "N"].sum()],
        })
        g = pd.concat([g.loc[~rare], other], ignore_index=True)

    total = g["N"].sum()
    g["pct_total"] = (g["N"] / total * 100).round(2)
    g["pct_needs_ml_true"] = (g["needs_ml_true"] / g["N"] * 100).round(2)

    other_pct = 0.0
    if (g["rule"] == "OTHER(<10)").any():
        other_pct = float(g.loc[g["rule"] == "OTHER(<10)", "pct_total"].iloc[0])
    g["pct_total_OTHER(<10)"] = round(other_pct, 2)

    out = g.rename(columns={"rule": rule_col})[
        [rule_col, "N", "pct_total", "needs_ml_true", "needs_ml_false", "pct_needs_ml_true", "pct_total_OTHER(<10)"]
    ]
    out.to_csv(out_csv, index=False)

    # stacked % plot
    plot_df = out.copy()
    plot_df["pct_true"] = plot_df["needs_ml_true"] / plot_df["N"] * 100
    plot_df["pct_false"] = plot_df["needs_ml_false"] / plot_df["N"] * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(plot_df[rule_col].astype(str), plot_df["pct_false"], label="needs_ml False")
    ax.bar(plot_df[rule_col].astype(str), plot_df["pct_true"], bottom=plot_df["pct_false"], label="needs_ml True")
    ax.set_ylabel("%")
    ax.set_title(out_png.stem)
    ax.legend(loc="best")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

coverage_table(
    df_full, "agent_rule", "needs_ml_agent",
    out_dir / "coverage_agent_rules.csv",
    out_dir / "coverage_agent_rules_stacked.png",
)

coverage_table(
    df_full, "patient_rule", "needs_ml_patient",
    out_dir / "coverage_patient_rules.csv",
    out_dir / "coverage_patient_rules_stacked.png",
)

print("Saved coverage outputs in:", out_dir)


Saved coverage outputs in: analyse\csv_heuristique_study


In [48]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)

def _safe_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred) if len(y_true) else np.nan

def _safe_macro(y_true, y_pred, metric_fn):
    if len(y_true) == 0:
        return np.nan
    labels = sorted(pd.unique(y_true))
    return metric_fn(y_true, y_pred, average="macro", labels=labels, zero_division=0)

def quality_by_rule(df, rule_col, pred_col, gold_col):
    rows = []
    for rule, g in df.groupby(rule_col, dropna=False):
        missing_mask = g[pred_col].isna() | (g[pred_col].astype(str).str.strip() == "")
        excluded = int(missing_mask.sum())

        g_valid = g.loc[~missing_mask].copy()
        y_true = g_valid[gold_col].astype(str)
        y_pred = g_valid[pred_col].astype(str)

        rows.append({
            rule_col: rule,
            "N": int(len(g)),
            "excluded_missing_pred": excluded,
            "accuracy": _safe_accuracy(y_true, y_pred),
            "macro_f1": _safe_macro(y_true, y_pred, f1_score),
            "macro_precision": _safe_macro(y_true, y_pred, precision_score),
            "macro_recall": _safe_macro(y_true, y_pred, recall_score),
        })

    return pd.DataFrame(rows).sort_values("N", ascending=False)

# Agent
agent_df = quality_by_rule(
    df_lab,
    rule_col="agent_rule",
    pred_col="agent_value",
    gold_col="agent label",
)
agent_df.to_csv(out_dir / "heuristic_quality_agent_by_rule.csv", index=False)

# Patient (keep "-" as class)
patient_df = quality_by_rule(
    df_lab,
    rule_col="patient_rule",
    pred_col="patient_value",
    gold_col="patient labels",
)
patient_df.to_csv(out_dir / "heuristic_quality_patient_by_rule.csv", index=False)

# Summary txt
def top_bottom(df, rule_col, name):
    df20 = df[df["N"] >= 20].copy()
    top5 = df20.sort_values("macro_f1", ascending=False).head(5)
    bot5 = df20.sort_values("macro_f1", ascending=True).head(5)
    lines = []
    lines.append(f"{name} (N>=20)")
    lines.append("Top 5 macro_f1:")
    lines.append(top5[[rule_col, "N", "macro_f1", "macro_precision", "macro_recall"]].to_string(index=False))
    lines.append("")
    lines.append("Bottom 5 macro_f1:")
    lines.append(bot5[[rule_col, "N", "macro_f1", "macro_precision", "macro_recall"]].to_string(index=False))
    lines.append("")
    return lines

summary_lines = []
summary_lines += top_bottom(agent_df, "agent_rule", "AGENT")
summary_lines += top_bottom(patient_df, "patient_rule", "PATIENT")

(out_dir / "heuristic_quality_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved:", out_dir / "heuristic_quality_agent_by_rule.csv")
print("Saved:", out_dir / "heuristic_quality_patient_by_rule.csv")
print("Saved:", out_dir / "heuristic_quality_summary.txt")


Saved: analyse\csv_heuristique_study\heuristic_quality_agent_by_rule.csv
Saved: analyse\csv_heuristique_study\heuristic_quality_patient_by_rule.csv
Saved: analyse\csv_heuristique_study\heuristic_quality_summary.txt


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)

def _to_bool(s):
    return s.astype(str).str.strip().str.upper().map({"TRUE": True, "FALSE": False}).fillna(False)

def coverage_table(df, rule_col, needs_col, out_csv, out_png, min_n=10):
    df = df.copy()
    df[needs_col] = _to_bool(df[needs_col])

    # counts
    g = df.groupby(rule_col, dropna=False)[needs_col].value_counts(dropna=False).unstack(fill_value=0)
    g = g.reindex(columns=[False, True], fill_value=0)
    g.columns = ["needs_ml_false", "needs_ml_true"]
    g["N"] = g["needs_ml_false"] + g["needs_ml_true"]
    g = g.reset_index().rename(columns={rule_col: "rule"})
    g = g.sort_values("N", ascending=False)

    # group rare rules
    rare = g["N"] < min_n
    if rare.any():
        other = pd.DataFrame({
            "rule": [f"OTHER(<{min_n})"],
            "needs_ml_false": [g.loc[rare, "needs_ml_false"].sum()],
            "needs_ml_true": [g.loc[rare, "needs_ml_true"].sum()],
            "N": [g.loc[rare, "N"].sum()],
        })
        g = pd.concat([g.loc[~rare], other], ignore_index=True)

    total = g["N"].sum()
    g["pct_total"] = (g["N"] / total * 100).round(2)
    g["pct_needs_ml_true"] = (g["needs_ml_true"] / g["N"] * 100).round(2)

    out = g.rename(columns={"rule": rule_col})[
        [rule_col, "N", "pct_total", "needs_ml_true", "needs_ml_false", "pct_needs_ml_true"]
    ]
    out.to_csv(out_csv, index=False)

    # stacked % plot
    plot_df = out.copy()
    plot_df["pct_true"] = plot_df["needs_ml_true"] / plot_df["N"] * 100
    plot_df["pct_false"] = plot_df["needs_ml_false"] / plot_df["N"] * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(plot_df[rule_col].astype(str), plot_df["pct_false"], label="needs_ml False")
    ax.bar(plot_df[rule_col].astype(str), plot_df["pct_true"],
           bottom=plot_df["pct_false"], label="needs_ml True")
    ax.set_ylabel("%")
    ax.set_title(out_png.stem)
    ax.legend(loc="best")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

# Agent rules
coverage_table(
    df_full, "agent_rule", "needs_ml_agent",
    out_dir / "coverage_agent_rules.csv",
    out_dir / "coverage_agent_rules_stacked.png",
    min_n=10
)

# Patient rules
coverage_table(
    df_full, "patient_rule", "needs_ml_patient",
    out_dir / "coverage_patient_rules.csv",
    out_dir / "coverage_patient_rules_stacked.png",
    min_n=10
)

print("Saved coverage outputs in:", out_dir)


Saved coverage outputs in: analyse\csv_heuristique_study


In [50]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)

def _safe_acc(y_true, y_pred):
    return accuracy_score(y_true, y_pred) if len(y_true) else np.nan

def _safe_macro_f1(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    labels = sorted(pd.unique(y_true))
    return f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)

def quality_by_rule(df, rule_col, pred_col, gold_col):
    rows = []
    for rule, g in df.groupby(rule_col, dropna=False):
        # exclude missing predictions
        missing = g[pred_col].isna() | (g[pred_col].astype(str).str.strip() == "")
        g_valid = g.loc[~missing]
        y_true = g_valid[gold_col].astype(str)
        y_pred = g_valid[pred_col].astype(str)
        rows.append({
            rule_col: rule,
            "N": int(len(g)),
            "excluded_missing_pred": int(missing.sum()),
            "accuracy": _safe_acc(y_true, y_pred),
            "macro_f1": _safe_macro_f1(y_true, y_pred),
        })
    return pd.DataFrame(rows).sort_values("N", ascending=False)

# Agent
agent_df = quality_by_rule(
    df_lab, "agent_rule", "agent_value", "agent label"
)
agent_df.to_csv(out_dir / "heuristic_quality_agent_by_rule.csv", index=False)

# Patient (keep "-" as class)
patient_df = quality_by_rule(
    df_lab, "patient_rule", "patient_value", "patient labels"
)
patient_df.to_csv(out_dir / "heuristic_quality_patient_by_rule.csv", index=False)

# Summary txt (top 5 / worst 5, N>=20)
def top_bottom(df, rule_col, name):
    df20 = df[df["N"] >= 20].copy()
    top5 = df20.sort_values("macro_f1", ascending=False).head(5)
    bot5 = df20.sort_values("macro_f1", ascending=True).head(5)
    lines = []
    lines.append(f"{name} (N>=20)")
    lines.append("Top 5 macro_f1:")
    lines.append(top5[[rule_col, "N", "macro_f1"]].to_string(index=False))
    lines.append("")
    lines.append("Bottom 5 macro_f1:")
    lines.append(bot5[[rule_col, "N", "macro_f1"]].to_string(index=False))
    lines.append("")
    return lines

summary_lines = []
summary_lines += top_bottom(agent_df, "agent_rule", "AGENT")
summary_lines += top_bottom(patient_df, "patient_rule", "PATIENT")

(out_dir / "summary_quality.txt").write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved:", out_dir / "heuristic_quality_agent_by_rule.csv")
print("Saved:", out_dir / "heuristic_quality_patient_by_rule.csv")
print("Saved:", out_dir / "summary_quality.txt")


Saved: analyse\csv_heuristique_study\heuristic_quality_agent_by_rule.csv
Saved: analyse\csv_heuristique_study\heuristic_quality_patient_by_rule.csv
Saved: analyse\csv_heuristique_study\summary_quality.txt


In [51]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)

# ensure discourse_margin exists
if "discourse_margin" not in df_full.columns:
    df_full = df_full.copy()
    df_full["discourse_max"] = df_full[["topic_conf", "last_agent_conf", "last_patient_conf"]].max(axis=1)
    df_full["discourse_second"] = df_full[["topic_conf", "last_agent_conf", "last_patient_conf"]].apply(
        lambda r: r.nlargest(2).iloc[-1], axis=1
    )
    df_full["discourse_margin"] = df_full["discourse_max"] - df_full["discourse_second"]

# normalize rule codes for plotting
for _col in ["patient_rule", "agent_rule"]:
    if _col in df_full.columns:
        df_full[_col] = df_full[_col].where(
            df_full[_col].isna(),
            df_full[_col].astype(str).str.replace("R23", "C2", regex=False)
        )

def _prepare_rules(df, rule_col, min_n=15):
    counts = df[rule_col].value_counts(dropna=False)
    keep = counts[counts >= min_n].index.tolist()
    df = df.copy()
    df[rule_col] = df[rule_col].where(df[rule_col].isin(keep), other=f"OTHER(<{min_n})")
    order = df[rule_col].value_counts().index.tolist()
    df[rule_col] = pd.Categorical(df[rule_col], categories=order, ordered=True)
    return df, order

# patient: discourse_margin
df_p, order_p = _prepare_rules(df_full, "patient_rule", min_n=15)
ax = df_p.boxplot(column="discourse_margin", by="patient_rule", vert=False, grid=False, figsize=(10, 6))
ax.set_title("discourse_margin by patient_rule (N>=15)")
ax.set_ylabel("patient_rule")
ax.set_xlabel("discourse_margin")
plt.suptitle("")
plt.tight_layout()
plt.savefig(out_dir / "patient_rule_boxplot_discourse_margin.png", dpi=150)
plt.close()

# patient: noun_count
ax = df_p.boxplot(column="noun_count", by="patient_rule", vert=False, grid=False, figsize=(10, 6))
ax.set_title("noun_count by patient_rule (N>=15)")
ax.set_ylabel("patient_rule")
ax.set_xlabel("noun_count")
plt.suptitle("")
plt.tight_layout()
plt.savefig(out_dir / "patient_rule_boxplot_noun_count.png", dpi=150)
plt.close()

# agent: discourse_margin
df_a, order_a = _prepare_rules(df_full, "agent_rule", min_n=15)
ax = df_a.boxplot(column="discourse_margin", by="agent_rule", vert=False, grid=False, figsize=(10, 6))
ax.set_title("discourse_margin by agent_rule (N>=15)")
ax.set_ylabel("agent_rule")
ax.set_xlabel("discourse_margin")
plt.suptitle("")
plt.tight_layout()
plt.savefig(out_dir / "agent_rule_boxplot_discourse_margin.png", dpi=150)
plt.close()

print("Saved boxplots in:", out_dir)


Saved boxplots in: analyse\csv_heuristique_study


In [52]:
import pandas as pd
from pathlib import Path

out_dir = Path("analyse") / "csv_heuristique_study"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "case_studies_rules.md"

# --- choose 3 rules: fallback + 2 strongest (macro_f1) ---
# assumes heuristic_quality_*_by_rule.csv already generated
qual_path = out_dir / "heuristic_quality_patient_by_rule.csv"
if not qual_path.exists():
    print("Missing:", qual_path)
else:
    qual = pd.read_csv(qual_path)

    # fallback rule: most frequent
    fallback_rule = qual.sort_values("N", ascending=False).iloc[0]["patient_rule"]

    # best rules with N>=20
    best_rules = qual[qual["N"] >= 20].sort_values("macro_f1", ascending=False)["patient_rule"].head(2).tolist()

    rules = [fallback_rule] + [r for r in best_rules if r != fallback_rule]
    rules = rules[:3]

    lines = []
    lines.append("# Case studies (rules)")
    lines.append("")

    for rule in rules:
        lines.append(f"## Rule: {rule}")
        subset = df_lab[df_lab["patient_rule"] == rule].copy()

        # mark correct / incorrect
        subset["correct"] = subset["patient_value"].astype(str) == subset["patient labels"].astype(str)

        # pick 1 correct + 1 incorrect if possible
        examples = []
        if (subset["correct"] == True).any():
            examples.append(subset[subset["correct"] == True].iloc[0])
        if (subset["correct"] == False).any():
            examples.append(subset[subset["correct"] == False].iloc[0])

        for ex in examples:
            lines.append("- clause_id: {}".format(ex.get("clause_id")))
            lines.append("  - contexte: {}".format(ex.get("contexte")))
            lines.append("  - pred vs gold: {} → {}".format(ex.get("patient_value"), ex.get("patient labels")))
            lines.append("  - needs_ml_patient: {}".format(ex.get("needs_ml_patient")))
            lines.append("  - features: discourse_margin={}, noun_count={}, verb_index={}, topic_conf={}, last_agent_conf={}".format(
                ex.get("discourse_margin"), ex.get("noun_count"), ex.get("verb_index"),
                ex.get("topic_conf"), ex.get("last_agent_conf")
            ))
        lines.append("")

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print("Saved:", out_path)


Saved: analyse\csv_heuristique_study\case_studies_rules.md
